# §34: K5df with `loss.dvs_anchor_mode=ignore` — **K5dfi**, and the same with `data.is_egocentric=true` — **K5dfie**, s2024

Rule: `core/docs/final/KIM_RUN.md` **§34**, written before any number. K5f is the reference (§31.1), K5df the control of the primary contrast. PRELIMINARY (n = 1). Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. `nexar_test` is never touched.

| row | DVS | `dvs_anchor_mode` | `is_egocentric` | here |
|---|---|---|---|---|
| **K5f** | off | (inert) | false | §31 run + reads, **read only** (reference) |
| **K5df** | on (θ 0.7 / 0.85, seam mask) | span | false | §32 run + reads, **read only** (control) |
| **K5dfi** | on | **ignore** | false | train + read |
| **K5dfie** | on | ignore | **true** (θ_ego 0.85, ≤ 2 clips per splice) | train + read |

**Why not "K5f + ignore":** K5f has DVS off (`theta = 1.0`), so no clip is ever spliced, no frame is pseudo-positive, and `ignore` masks nothing (`test_ignore_positive_is_a_noop_on_an_all_normal_row`).

**R4 (fixed before any number, margin 0.01):** GAIN iff the contrast is ≥ +0.01 with the interval above 0 on DoTA-CAP-dev or DADA and a COST on none of Nexar val / DoTA-CAP-dev / DADA / DAD; COST iff ≤ −0.01 with the interval below 0 on any of them; else NO DIFFERENCE. Primary **K5dfi − K5df**; secondary **K5dfie − K5dfi**. Candidate iff the row − K5f is above on ≥ 1 set and below on none; K5f stays the reference unless the user decides.

**Guards:** J10 (step 1,749), config diff to K5df = `loss.dvs_anchor_mode` (K5dfi) / + `data.is_egocentric` (K5dfie), last-epoch `mul_mil` > 0.01, K5dfi's step-1 `dvs_sup` differs from K5df's (the mask is live; the report raises otherwise) and `mil` / `dvs_sup_mil` equal (same draws; a flag, not a stop). K5dfie has no matched start.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, preflight (K5f, K5df, bakes, census) | no | 2-5 min |
| 4 | pytest on the touched code — **raises if red** | no | 2-3 min |
| 5 | `Nexar_window` + the §31 fix on its real meta | no | < 1 min |
| 6 | train K5dfi, K5dfie (each about as long as K5) + checks | yes | 2 × K5 |
| 7 | zero-shot reads: DoTA-CAP-dev, DADA, TAD, DAD | yes | 2 × 20-30 min |
| 8 | `nexar_val` reads with `--save-scores` | yes | 2 × 10-20 min |
| 9 | `k5_dvs_ignore_report` -> **paste `REPORTS/k5dfi_report.md` to Claude** | no | 1 min |

**Watch the log: past 2 s/step, stop and paste it.** A re-run skips what is on Drive. Everything new goes to `Thesis-Final/outputs/kim_dvs_ignore/`.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, A2 bakes, Nexar caches, census
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K5f, K5df and their reads, DAD; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 34. ' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §34 -- re-upload kat-vad/'
assert 'if not is_abnormal:\n            return NORMAL_CLASS' in (REPO / 'core/data/dataset.py').read_text(), \
    'core/data/dataset.py has no §31 fix -- re-upload kat-vad/ (training now would repeat §30)'
assert (REPO / 'core/tools/k5_dvs_ignore_report.py').is_file(), 'no k5_dvs_ignore_report -- re-upload kat-vad/'
assert 'KIM_DVS_IGNORE_ROW' in (REPO / 'core/constants.py').read_text(), 'constants has no §34 rows -- re-upload kat-vad/'
assert '--save-scores' in (REPO / 'core/tools/nexar_eval.py').read_text(), 'nexar_eval has no --save-scores -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5       = constants.KIM_MAIN_ARM                  # "K5": names the eval sub-folders only (eval/<set>/K5)
K5F      = constants.KIM_REFERENCE_ARM             # K5f: the reference (§31.1)
K5DF     = constants.KIM_FIXED_DVS_ROW             # K5df: the control of the primary contrast (§32)
K5DFI, K5DFIE = constants.KIM_DVS_IGNORE_ROW, constants.KIM_DVS_IGNORE_EGO_ROW
NEW_ROWS = (K5DFI, K5DFIE)
SEED     = constants.KIM_SEED                      # 2024
ENC      = constants.V5_ENCODER                    # V2-S
DATASET  = constants.NEXAR_DATASET
C_WINDOW = constants.NEXAR_CONSTRUCTION_WINDOW
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
NEXAR_SPLIT = constants.V2_SPLIT_NEXAR_VAL         # nexar_test is never read here
TAD_NAME = constants.TAD_DATASET
DAD_NAME, DAD_STRIDE = constants.DAD_DATASET, constants.DAD_STRIDE

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
WINDOW   = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C_WINDOW}'     # K5's corpus: the reference of every assert
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'                # nexar_train statistics, every train video
# K5f: the reference (read only)
K5F_ROOT = constants.OUTPUT_ROOT / constants.KIM_FIXED_OUTPUT_DIRNAME / K5F
K5F_RUN  = K5F_ROOT / K5 / f's{SEED}'
# K5df: the control (read only), a row of §32's folder
K5DF_ROOT = constants.OUTPUT_ROOT / constants.KIM_LNEG_CAP_FIXED_OUTPUT_DIRNAME / K5DF
K5DF_RUN  = K5DF_ROOT / K5 / f's{SEED}'
# Zero-shot inputs: the bakes K5 was read on
A2_DADA  = V2_CACHE / 'v2' / 'A2_S' / f'{T2NAME}_test_sources_nexar'
A2_DOTA  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
A2_TAD   = V2_CACHE / 'v2' / 'A2_S' / f'{TAD_NAME}_s8_nexar'
A2_DAD   = constants.CACHE_ROOT / 'v2' / 'A2_S' / f'{DAD_NAME}_s{DAD_STRIDE}_nexar'
T2       = constants.DATA_ROOT / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
TAD      = constants.DATA_ROOT / TAD_NAME
DAD_DATA = Path(DRIVE_F) / 'data' / f'{DAD_NAME}_s{DAD_STRIDE}'
# Nexar
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
NEXAR_X  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
NEXAR_U  = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
KNN      = V2_CACHE / 'knn' / f'{DATASET}_{C_WINDOW}' / constants.KNN_CACHE_FILENAME
# Outputs: one folder per row, each read like K5 (eval/<set>/K5, eval/nexar/<row>)
ROOT     = constants.OUTPUT_ROOT / constants.KIM_DVS_IGNORE_OUTPUT_DIRNAME
DATA     = WINDOW                                                # K5's own corpus, read-only
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/k5_dvs_ignore')
LOCAL.mkdir(parents=True, exist_ok=True)


def row_paths(row):
    run_dir = ROOT / row / K5 / f's{SEED}'
    return run_dir, run_dir / 'checkpoint_last.pt', ROOT / row / 'eval'


INPUTS = (('Nexar_window labels', WINDOW / constants.LABELS_TRAIN_FILENAME), ('Nexar A2 bake', A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME),
          ('K5f checkpoint', K5F_RUN / 'checkpoint_last.pt'), ('K5f verified', K5F_RUN / 'verified.json'),
          ('K5f config', K5F_RUN / 'config.yaml'), ('K5f metrics', K5F_RUN / 'metrics.jsonl'),
          ('K5f nexar_val read', K5F_ROOT / 'eval' / 'nexar' / K5F / 'nexar_readout.json'),
          ('K5f DoTA read', K5F_ROOT / 'eval' / 'dota' / K5), ('K5f DADA read', K5F_ROOT / 'eval' / 'dada' / K5),
          ('K5f TAD read', K5F_ROOT / 'eval' / 'tad' / K5 / f's{SEED}' / 'results.json'),
          ('K5f DAD read', K5F_ROOT / 'eval' / constants.DAD_OUTPUT_SET / K5 / f's{SEED}' / 'results.json'),
          ('K5df checkpoint', K5DF_RUN / 'checkpoint_last.pt'), ('K5df verified', K5DF_RUN / 'verified.json'),
          ('K5df config', K5DF_RUN / 'config.yaml'), ('K5df metrics', K5DF_RUN / 'metrics.jsonl'),
          ('K5df nexar_val read', K5DF_ROOT / 'eval' / 'nexar' / K5DF / 'nexar_readout.json'),
          ('K5df DoTA read', K5DF_ROOT / 'eval' / 'dota' / K5), ('K5df DADA read', K5DF_ROOT / 'eval' / 'dada' / K5),
          ('K5df TAD read', K5DF_ROOT / 'eval' / 'tad' / K5 / f's{SEED}' / 'results.json'),
          ('K5df DAD read', K5DF_ROOT / 'eval' / constants.DAD_OUTPUT_SET / K5 / f's{SEED}' / 'results.json'),
          ('KNN cache', KNN), ('DADA A2 bake', A2_DADA), ('DoTA-CAP A2 bake', A2_DOTA), ('TAD A2 bake', A2_TAD), ('DAD A2 bake', A2_DAD),
          ('T2 meta', T2 / constants.META_FILENAME), ('T2 census', T2_CENSUS), ('DADA annotation', ANNOT),
          ('DoTA s1', DOTA_S1), ('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME),
          ('DAD labels', DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME),
          ('Nexar census', CENSUS), ('Nexar CLIP s1', NEXAR_X), ('Nexar V2-S s1', NEXAR_U))
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | corpus {DATA} (K5\'s) | runs/reads -> {ROOT}')


In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 6-8 need one'
df -h /content | tail -1

In [ ]:
import hashlib  # noqa: E402
import torch
import yaml

from core.config import Config                       # noqa: E402
from core.data.v2_splits import load_split           # noqa: E402
from core.tools.dada_whole_eval import test_sources  # noqa: E402
from core.tools.v5_stage1 import _flat               # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(drive_dir, steps):
    rec, ckpt = drive_dir / 'verified.json', drive_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())



# Preflight on small files only (C39)
k5f_cfg = _flat(yaml.safe_load((K5F_RUN / 'config.yaml').read_text()))
assert k5f_cfg['kim.enabled'] and k5f_cfg['kim.prototypes'] and k5f_cfg['kim.edge_mask'], 'K5F_RUN is not K5f'
assert k5f_cfg['dvs.theta'] == 1.0 and k5f_cfg['data.dataset'] == DATASET, 'K5F_RUN is not K5f on Nexar'
STEPS = json.loads((K5F_RUN / 'verified.json').read_text())['step']             # 1,749
EPOCHS = k5f_cfg['train.num_epochs']                                            # 33
assert is_verified(K5F_RUN, STEPS) and metrics_last(K5F_RUN / 'metrics.jsonl')['global_step'] == STEPS, 'K5f: J10'
k5df_cfg = _flat(yaml.safe_load((K5DF_RUN / 'config.yaml').read_text()))
assert k5df_cfg['kim.seam_mask'] and k5df_cfg['dvs.theta'] == 0.7 and k5df_cfg['dvs.theta_ego'] == 0.85, 'K5DF_RUN is not K5df'
assert k5df_cfg['loss.dvs_anchor_mode'] == constants.DVS_ANCHOR_MODE_SPAN and not k5df_cfg['data.is_egocentric'], \
    'K5df must be the span / non-ego control'
assert is_verified(K5DF_RUN, STEPS) and metrics_last(K5DF_RUN / 'metrics.jsonl')['global_step'] == STEPS, 'K5df: J10'
SHA = manifest(A2_NEXAR)['train_ids_sha1']
for name, d in (('Nexar', A2_NEXAR), ('DADA', A2_DADA), ('DoTA-CAP', A2_DOTA), ('TAD', A2_TAD), ('DAD', A2_DAD)):
    m = manifest(d)
    assert (m['crn'], m['motion'], m['train_ids_sha1']) == (constants.V2_OFF, ENC, SHA), f'{name} bake: {m}'
census = json.loads(CENSUS.read_text())['videos']
TRAIN_IDS = sorted(load_split(constants.V2_SPLIT_NEXAR_TRAIN))
NEG_IDS = [v for v in TRAIN_IDS if census[v]['label'] == 0]
DADA_IDS = test_sources(T2)
CAP_IDS = sorted(load_split(CAP_DEV))
NEXAR_IDS = sorted(load_split(NEXAR_SPLIT))
TAD_IDS = sorted(json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
DAD_IDS = sorted(json.loads((DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert (len(TRAIN_IDS), len(NEG_IDS)) == (900, 449), (len(TRAIN_IDS), len(NEG_IDS))
assert len(DADA_IDS) == 370 and len(TAD_IDS) == constants.V2_TADZ_VIDEOS and len(DAD_IDS) == constants.DAD_TEST_VIDEOS
for name, d, ids in (('Nexar train bake', A2_NEXAR, TRAIN_IDS), ('DADA bake', A2_DADA, DADA_IDS),
                     ('DoTA-CAP bake', A2_DOTA, CAP_IDS), ('TAD bake', A2_TAD, TAD_IDS), ('DAD bake', A2_DAD, DAD_IDS),
                     ('Nexar CLIP', NEXAR_X, NEXAR_IDS), ('Nexar V2-S', NEXAR_U, NEXAR_IDS)):
    miss = [v for v in ids if not (d / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} of {name} missing: {miss[:5]}'
print(f'preflight OK: K5f / K5df step {STEPS} ({EPOCHS} epochs) | train {len(TRAIN_IDS)} ({len(NEG_IDS)} negative) | '
      f'DoTA-CAP-dev {len(CAP_IDS)}, DADA {len(DADA_IDS)}, TAD {len(TAD_IDS)}, DAD {len(DAD_IDS)}, {NEXAR_SPLIT} {len(NEXAR_IDS)}')


## 4. Tests on the touched code (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_baseline_losses.py',
     'core/tests/test_dvs_dataset.py', 'core/tests/test_k5_dvs_ignore_report.py', 'core/tests/test_kim.py'])

## 5. K5's corpus `Nexar_window` (read-only) and the §31 fix on its real meta

In [ ]:
labels = json.loads((DATA / constants.LABELS_TRAIN_FILENAME).read_text())
windows = json.loads((DATA / constants.WINDOWS_FILENAME).read_text())
n_pos = sum(labels.values())
assert (len(labels), n_pos) == (7311, 1689), f'not K5\'s corpus: {len(labels)} windows, {n_pos} positive'
sources = {windows[w]['source'] for w in labels}
assert not sources & set(NEG_IDS), 'Nexar_window holds a negative-video window -- that is K5neg / K5nc2, not K5'
assert sources <= set(TRAIN_IDS), 'a window comes from outside nexar_train'
steps_per_epoch = -(-2 * n_pos // 64)            # balanced epoch = 2 x positives, batch 64
assert steps_per_epoch * EPOCHS == STEPS, f'{steps_per_epoch} x {EPOCHS} != K5 {STEPS}'
print(f'Nexar_window: {n_pos} positive + {len(labels) - n_pos} normal windows from {len(sources)} positive videos | '
      f'{steps_per_epoch} steps/epoch x {EPOCHS} = {STEPS}')

# §31: the bug on the real meta (how many label-0 windows carry a non-Normal class), then the fix
from types import SimpleNamespace  # noqa: E402

from core.data.dataset import DVSFeatureDataset  # noqa: E402
from core.data.msad import NORMAL_CLASS  # noqa: E402

meta = json.loads((DATA / constants.META_FILENAME).read_text())
bad_meta = sum(1 for w, y in labels.items() if y == 0 and meta[w]['class_name'] != NORMAL_CLASS)
fake = SimpleNamespace(_classes={w: m['class_name'] for w, m in meta.items()})
got = {w: DVSFeatureDataset._class_name(fake, w, bool(y)) for w, y in labels.items()}
still_bad = [w for w, y in labels.items() if y == 0 and got[w] != NORMAL_CLASS]
pos_classes = sorted({got[w] for w, y in labels.items() if y})
assert not still_bad, f'{len(still_bad)} label-0 windows still non-Normal: {still_bad[:5]} -- the fix is not in this code'
print(f'meta: {bad_meta} / {len(labels) - n_pos} label-0 windows carry a non-Normal class (what K5 trained on) | '
      f'fixed path: 0 non-Normal; positives -> {pos_classes}')


## 6. Train K5dfi and K5dfie (§34)

**Watch the log: past 2 s/step, stop and paste it.** Each run: J10, the config diff against K5df (K5dfi: `loss.dvs_anchor_mode`; K5dfie: + `data.is_egocentric`), the fix check (`mul_mil`), and the step-1 start check (K5dfi only).

In [ ]:
from IPython.display import Markdown, display  # noqa: E402

K5D_SETS = ['kim.enabled=true', 'kim.prototypes=true', 'kim.edge_mask=true', 'kim.seam_mask=true',
            'dvs.theta=0.7', 'dvs.theta_ego=0.85']
BASE = ['train.stage=2', f'train.seed={SEED}', f'train.num_epochs={EPOCHS}', 'train.amp=true',
        f'data.dataset={DATASET}', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
        'kip.enabled=false', 'loss.dvs_anchor_mode=span', 'data.is_egocentric=false',
        f'v2.motion={ENC}', f'model.motion_dim={constants.V5_GRID_DIM}',
        f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}']
L_KNN = ['--knn-cache', stage(KNN, LOCAL / 'stage' / 'knn' / KNN.name)]
IGN = constants.DVS_ANCHOR_MODE_IGNORE
# row: its overrides of K5df's command (key -> value)
ARMS = {K5DFI: {'loss.dvs_anchor_mode': IGN},
        K5DFIE: {'loss.dvs_anchor_mode': IGN, 'data.is_egocentric': 'true'}}
DEFAULTS = _flat(Config().to_dict())


def sets_for(row):
    '''K5df's --set list with the row's overrides replacing the same keys (never a second copy of a key).'''
    over = ARMS[row]
    out = [s for s in (*BASE, *K5D_SETS) if s.split('=', 1)[0] not in over]
    return [*out, *[f'{k}={v}' for k, v in over.items()]]


def config_diff(base, got):
    '''Keys that differ; a key added after ``base`` trained may sit at its default.'''
    out = {}
    for k in set(base) | set(got):
        if base.get(k) == got.get(k):
            continue
        if k not in base and got.get(k) == DEFAULTS.get(k):
            continue
        if k not in got and base.get(k) == DEFAULTS.get(k):
            continue
        out[k] = (base.get(k), got.get(k))
    return out


def check_run(arm):
    run_dir, ckpt, _ = row_paths(arm)
    payload = torch.load(ckpt, map_location='cpu', weights_only=False)
    assert payload['global_step'] == metrics_last(run_dir / 'metrics.jsonl')['global_step'] == STEPS, f'{arm}: J10'
    got = _flat(yaml.safe_load((run_dir / 'config.yaml').read_text()))
    diff = config_diff(k5df_cfg, got)
    want = {'loss.dvs_anchor_mode', *(k for k in ARMS[arm] if k != 'loss.dvs_anchor_mode')}
    assert set(diff) == want, f'{arm} must differ from K5df in exactly {sorted(want)}: {diff}'
    assert diff['loss.dvs_anchor_mode'][1] == IGN, diff
    if 'data.is_egocentric' in want:
        assert diff['data.is_egocentric'][1] is True, diff
    rows = [json.loads(l) for l in (run_dir / 'metrics.jsonl').read_text().splitlines() if l.strip()]
    last_rows = [r for r in rows if r['epoch'] == rows[-1]['epoch']]
    mul_last = sum(r['mul_mil'] for r in last_rows) / len(last_rows)
    assert mul_last > constants.KIM_FIXED_MUL_MIL_FLOOR, f'{arm}: last-epoch mul_mil {mul_last:.4f} -- trained on the pre-§31 code'
    print(f'{arm} config diff to K5df: {diff} | fix check: last-epoch mul_mil {mul_last:.4f} > {constants.KIM_FIXED_MUL_MIL_FLOOR}')
    ref = [json.loads(l) for l in (K5DF_RUN / 'metrics.jsonl').read_text().splitlines() if l.strip()][0]
    print(f"{arm} step 1: mil {rows[0]['mil']:.6f} (K5df {ref['mil']:.6f}) | dvs_sup {rows[0]['dvs_sup']:.6f} (K5df {ref['dvs_sup']:.6f}) | "
          f"dvs_sup_mil {rows[0]['dvs_sup_mil']:.6f} (K5df {ref['dvs_sup_mil']:.6f})")
    if arm == K5DFI:
        same = abs(rows[0]['mil'] - ref['mil']) <= constants.KIM_START_TOL
        print('G3 start check:', 'PASS (same draws)' if same else 'FLAG: step-1 mil differs from K5df -- report to Claude (same-runtime K5df retrain = option C)')
    else:
        print('K5dfie: no matched start (theta_ego / delta_m_ego change the splice draws); step-1 numbers printed only')


def train_arm(arm):
    run_dir, ckpt, _ = row_paths(arm)
    if is_verified(run_dir, STEPS):
        print(f'{arm} s{SEED}: verified on Drive -- not retrained')
        check_run(arm)
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'runs' / arm
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in sets_for(arm) for x in ('--set', s)],
         '--data-dir', stage(DATA, LOCAL / 'stage' / 'data'), '--clip-dir', stage(A2_NEXAR, LOCAL / 'stage' / 'a2_nexar'),
         '--output-dir', local, *L_KNN, *resume], log=LOCAL / f'train_{arm}.log')
    last = metrics_last(local / 'metrics.jsonl')
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{arm}: J10 locally {last["global_step"]}'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, run_dir / name)
    flush_drive()
    assert ckpt_step(ckpt) == STEPS, f'{arm}: Drive copy is not step {STEPS}'
    (local / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': ckpt.stat().st_size}))
    publish(local / 'verified.json', run_dir / 'verified.json')
    print(f'{arm}: trained in {(time.time() - t0) / 60:.0f} min | mil {last.get("mil")} | kim_mil {last.get("kim_mil")}')
    check_run(arm)


for arm in NEW_ROWS:
    print(arm, 'sets:', sets_for(arm)[-3:])
    train_arm(arm)


## 7. Zero-shot reads of K5dfi, K5dfie: DoTA-CAP-dev (protocol B), DADA (whole), TAD, DAD — K5f's protocols and bakes

In [ ]:
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
L_TAD = LOCAL / 'TAD_meta'
L_TAD.mkdir(parents=True, exist_ok=True)
for f in TAD.iterdir():                     # json / txt only -- never a frame folder
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_TAD / f.name)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in WINDOW.glob('*.json'):             # the definitions every Nexar row read with
    shutil.copy2(f, L_DEFS / f.name)


def read_evaluate(row, name, dataset, data_dir, bake, ids, out, ckpt):
    if not (out / 'results.json').exists():
        local = LOCAL / 'eval' / row / name
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', ckpt, '--set', f'data.dataset={dataset}',
             '--data-dir', data_dir, '--clip-dir', stage(bake, LOCAL / 'stage' / f'{name}_bake', ids),
             '--score-norm', 'auto', '--save-scores', '--output-dir', local], log=LOCAL / f'{row}_{name}.log')
        publish(local, out)
    r = json.loads((out / 'results.json').read_text())
    assert r['score_norm'] == 'none' and r['num_videos'] == len(ids), r
    print(f"{row} {name}: macro {r['auc_macro']:.4f} over {r['auc_macro_videos']} | micro {r['auc']:.4f} | AP {r['ap']:.4f}")


def read_zero_shot(row):
    _, ckpt, ev = row_paths(row)
    out = ev / 'dota' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dota'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', ckpt,
             '--input-dir', stage(A2_DOTA, LOCAL / 'stage' / 'cap', CAP_IDS), '--s1-dir', stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS),
             '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
             '--data-dir', stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels'), '--split', CAP_DEV,
             '--out-dir', local, '--device', 'auto'], log=LOCAL / f'{row}_dota.log')
        publish(local, out)
    print(f"{row} DoTA-CAP-dev: macro {json.loads((out / 'protocol_b_readout.json').read_text())['macro']['mean']:.4f}")
    out = ev / 'dada' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dada'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', ckpt, '--t2-dir', L_T2,
             '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', stage(A2_DADA, LOCAL / 'stage' / 'dada', DADA_IDS),
             '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / f'{row}_dada.log')
        publish(local, out)
    print(f"{row} DADA whole: macro {json.loads((out / 'dada_readout.json').read_text())['macro']['mean']:.4f}")
    read_evaluate(row, 'tad', TAD_NAME, L_TAD, A2_TAD, TAD_IDS, ev / 'tad' / K5 / f's{SEED}', ckpt)
    read_evaluate(row, 'dad', DAD_NAME, stage(DAD_DATA, LOCAL / 'stage' / 'dad_meta'), A2_DAD, DAD_IDS,
                  ev / constants.DAD_OUTPUT_SET / K5 / f's{SEED}', ckpt)
    flush_drive()


for row in NEW_ROWS:
    read_zero_shot(row)

## 8. `nexar_val` reads of K5dfi, K5dfie with `--save-scores` (`--final` is never passed)

In [ ]:
from core.tools.nexar_eval import ITEM_AUCS_JSON, READOUT_JSON, VIDEO_STATS_JSON  # noqa: E402


def read_nexar(row):
    _, ckpt, ev = row_paths(row)
    out = ev / 'nexar' / row
    done = all((out / f).exists() for f in (READOUT_JSON, ITEM_AUCS_JSON, VIDEO_STATS_JSON)) and \
        len(list((out / 'scores').glob('*.npz'))) == len(NEXAR_IDS)
    if not done:
        L_X = stage(NEXAR_X, LOCAL / 'stage' / 'nexar_x', NEXAR_IDS)
        L_U = stage(NEXAR_U, LOCAL / 'stage' / 'nexar_u', NEXAR_IDS)
        L_STATS = stage(A2_NEXAR, LOCAL / 'stage' / 'nexar_stats', [])     # manifest + statistics only
        local = LOCAL / 'eval' / row / 'nexar'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.nexar_eval', '--run', f's{SEED}', ckpt, '--stats-dir', L_STATS,
             '--video-s1-dir', L_U, '--census', CENSUS, '--clip-s1-dir', L_X,
             '--data-dir', L_DEFS, '--split', NEXAR_SPLIT, '--save-scores', '--out-dir', local], log=LOCAL / f'{row}_nexar.log')
        publish(local, out)
        flush_drive()
    r = json.loads((out / READOUT_JSON).read_text())
    assert r['split'] == NEXAR_SPLIT and r['videos'] == len(NEXAR_IDS), (r['split'], r['videos'])
    assert r['global_steps'] == {f's{SEED}': STEPS} and Path(r['checkpoints'][f's{SEED}']) == ckpt, (r['global_steps'], r['checkpoints'])
    print(f"{row}: crops macro {r['crops']['mean_over_placements']['macro']['mean']:.4f} | "
          f"whole macro {r['whole']['macro']['mean']:.4f} | clip AUC {r['whole']['clip_auc_max']:.4f}")


for row in NEW_ROWS:
    read_nexar(row)

## 9. The §34 report (K5f reference, K5df control, K5dfi, K5dfie) -> **paste `REPORTS/k5dfi_report.md` to Claude**

In [ ]:
local = LOCAL / 'report'
K5F_NEXAR = K5F_ROOT / 'eval' / 'nexar' / K5F
run([sys.executable, '-m', 'core.tools.k5_dvs_ignore_report', '--kim-root', K5F_ROOT, '--k5f-nexar-dir', K5F_NEXAR,
     '--df-root', K5DF_ROOT.parent, '--ign-root', ROOT, '--seed', SEED, '--out-dir', local])
publish(local, REPORTS)
print((REPORTS / f'{constants.KIM_DVS_IGNORE_REPORT_STEM}.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/k5dfi_report.md to Claude')
